# 03. Fitting, Interpreting, and Testing a Multiple Regression Model

A fitted multiple regression table gives several different pieces of evidence. This notebook separates them into coefficient-level inference and model-level inference.

By the end of this notebook, you should be able to:

- read a multiple-regression coefficient table;
- conduct coefficient-level t tests and confidence intervals;
- use the overall F test, ANOVA decomposition, $R^2$, and adjusted $R^2$ without confusing their roles.


In [1]:
from lite_setup import ensure_packages
await ensure_packages()

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


Running outside JupyterLite; assuming packages are already installed.


In [2]:
ads = pd.read_csv("data/advertising_sales.csv")
model = smf.ols("Sales ~ RadioSpend + PrintSpend", data=ads).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  Sales   R-squared:                       0.964
Model:                            OLS   Adj. R-squared:                  0.959
Method:                 Least Squares   F-statistic:                     224.5
Date:                Mon, 18 May 2026   Prob (F-statistic):           6.00e-13
Time:                        01:53:59   Log-Likelihood:                -25.465
No. Observations:                  20   AIC:                             56.93
Df Residuals:                      17   BIC:                             59.92
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      9.7542      0.583     16.731      0.0

## Coefficient Interpretation

For this fitted model:

- the `RadioSpend` coefficient is the expected change in sales for one additional unit of radio advertising spend, holding print advertising spend fixed;
- the `PrintSpend` coefficient is the expected change in sales for one additional unit of print advertising spend, holding radio advertising spend fixed;
- the intercept is the fitted sales value when both advertising spends are zero, which may or may not be operationally meaningful depending on whether zero spend is in the relevant data range.

The phrase "holding the other predictors fixed" is what makes multiple regression different from fitting separate simple regressions.

In [3]:
coef_table = pd.DataFrame({
    "estimate": model.params,
    "std_error": model.bse,
    "t_stat": model.tvalues,
    "p_value": model.pvalues,
})
coef_table


,estimate,std_error,t_stat,p_value
Intercept,9.754192,0.583001,16.730993,5.408300e-12
RadioSpend,1.464652,0.165568,8.846248,9.044851e-08
PrintSpend,2.302705,0.183024,12.581466,4.862616e-10


## Individual t Tests

For predictor $j$, the usual test is

$$H_0:\beta_j=0 \quad \text{versus} \quad H_a:\beta_j\ne 0,$$

using

$$t=\frac{b_j}{s\{b_j\}}$$

with $n-(k+1)$ degrees of freedom.

The same calculation also appears in the coefficient table for the intercept. The intercept test is mathematically valid, but its practical interpretation depends on whether all predictors equal to zero is meaningful.


A 100$(1-\alpha)$% confidence interval for $\beta_j$ is

$$b_j \pm t_{1-\alpha/2,\,n-k-1}\,s\{b_j\}.$$

Be careful when reading many individual coefficient tests at once. Each test has its own Type I error probability, so a long list of separate tests should not be treated as one perfectly controlled model-level decision.


In [4]:
alpha = 0.05
ci = model.conf_int(alpha=alpha)
ci.columns = ["lower", "upper"]
ci


,lower,upper
Intercept,8.524167,10.984218
RadioSpend,1.115335,1.813969
PrintSpend,1.916559,2.688852


## Overall F Test

The overall regression test asks whether the model with predictors improves over an intercept-only model:

$$H_0:\beta_1=\beta_2=\cdots=\beta_k=0.$$

This is different from asking whether every individual coefficient is significant. The F test is a joint test.


For $k$ predictors,

$$SST=SSR+SSE,$$

$$MSR=\frac{SSR}{k}, \qquad MSE=\frac{SSE}{n-k-1},$$

and the overall test statistic is

$$F_0=\frac{MSR}{MSE} \sim F_{k,\,n-k-1}\quad \text{under }H_0.$$


In [5]:
anova_table = sm.stats.anova_lm(model, typ=1)
anova_table


,df,sum_sq,mean_sq,F,PR(>F)
RadioSpend,1.0,255.490869,255.490869,290.624955,3.999276e-12
PrintSpend,1.0,139.156980,139.156980,158.293293,4.862616e-10
Residual,17.0,14.944845,0.879109,NaN,NaN


The same overall F test is also stored directly on the fitted model object. This is the most direct pattern when a problem asks for the overall F statistic and p-value.

In [6]:
print(f"Overall F statistic: {model.fvalue:.4f}")
print(f"Overall F-test p-value: {model.f_pvalue:.6g}")


Overall F statistic: 224.4591
Overall F-test p-value: 6.00044e-13


In [7]:
from checks import model_snapshot
model_snapshot(model)


{'n': 20,
 'df_resid': 17.0,
 'r_squared': 0.9635129122833551,
 'adj_r_squared': 0.9592203137284557,
 'f_pvalue': 6.000441503752843e-13}

## R-squared and Adjusted R-squared

$R^2$ is the fraction of total response variation explained by the fitted model. It cannot decrease when you add predictors, so it is not a sufficient model-selection rule. Adjusted $R^2$ includes a penalty for adding predictors and is usually more useful for comparing models with different numbers of predictors.


In [8]:
print(f"R-squared: {model.rsquared:.4f}")
print(f"Adjusted R-squared: {model.rsquared_adj:.4f}")


R-squared: 0.9635
Adjusted R-squared: 0.9592
